# Cluster para a Base Ativa

## Objetivos:

1. Definir as variáveis que entraram na análise do cluster;
2. Definir o número de clusters;
3. Comparar a taxa de churn para cada cluster.

**Finalidade**: O objetivo é encontrar clusters que possam ter diferentes comportamentos no que tange a taxa de churn. A partir disso, é possível desenvolver estratégias customizadas de rentabilização, retenção e atendimento. 

**Imports**

In [1]:
#1. Biblioteca Padrão
import warnings

#2. Bibliotecas de Terceiros

#Manipulação de dados
import numpy as np
import pandas as pd

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Estatística e Machine Learning
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from yellowbrick.cluster import KElbowVisualizer

# 3. Configurações
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

**Fórmula do KS**

In [2]:
def calcular_ks_2samp(df, alvo, escore):
    # Separa os escores para o grupo 'bons' e remove os valores ausentes
    bons_escore = df.loc[df[alvo] == 0, escore].dropna().rename('bons')
    
    # Separa os escores para o grupo 'maus' e remove os valores ausentes
    maus_escore = df.loc[df[alvo] == 1, escore].dropna().rename('maus')
    
    # Verifica se os grupos não estão vazios antes de calcular
    if bons_escore.empty or maus_escore.empty:
        print("Um dos grupos está vazio após remover valores ausentes.")
        return 0  # Retorna 0 ou outro valor indicando que não foi possível calcular

    # Calcula e retorna a estatística KS
    return stats.ks_2samp(bons_escore, maus_escore).statistic

# 1 - Visualização dos dados

## Base ativa

In [3]:
# Abrir o arquivo em parquet
df = pd.read_parquet('base_segmentacao_completa_abr_mai_2025.parquet', engine='fastparquet')

print("✅ Arquivo carregado com sucesso!")
print(f"📊 Shape: {df.shape}")
print(f"🗂️  Colunas: {list(df.columns)}")

✅ Arquivo carregado com sucesso!
📊 Shape: (1987106, 108)
🗂️  Colunas: ['data_movimento', 'tipo_movimento', 'flg_primeiro_plano', 'flg_funcionario', 'flg_permuta', 'flg_migrado', 'flg_desconto', 'num', 'id_planos_usuarios', 'ativo', 'data_contrato', 'data_cadastro', 'meses_base', 'dsc_categoria', 'dsc_tecnologia', 'plano', 'vencimento', 'valor', 'desconto', 'motivo_desconto', 'faixa_desconto', 'status_plano', 'download', 'upload', 'vlr_scm_plano_internet', 'vlr_sva_plano_internet', 'pppoe', 'descricao_plano', 'tipo_documento', 'nome', 'documento_original', 'cod_ibge', 'dsc_municipio_ibge', 'cod_ibge_cto', 'dsc_municipio_ibge_cto', 'posicao_gpon', 'concentrador', 'porta', 'caixa', 'posicao', 'lat_cto', 'long_cto', 'login', 'qtd_plano_internet', 'vlr_plano_internet', 'qtd_plano_sva', 'vlr_plano_sva', 'qtd_plano_hospedagem', 'vlr_plano_hospedagem', 'qtd_plano_radio', 'vlr_plano_radio', 'qtd_plano_telefone', 'vlr_plano_telefone', 'qtd_plano_mvno', 'vlr_plano_mvno', 'qtd_plano_outros', 'vlr_

In [4]:
df.head(3)

,data_movimento,tipo_movimento,flg_primeiro_plano,flg_funcionario,flg_permuta,flg_migrado,flg_desconto,num,id_planos_usuarios,ativo,data_contrato,data_cadastro,meses_base,dsc_categoria,dsc_tecnologia,plano,vencimento,valor,desconto,motivo_desconto,faixa_desconto,status_plano,download,upload,vlr_scm_plano_internet,vlr_sva_plano_internet,pppoe,descricao_plano,tipo_documento,nome,documento_original,cod_ibge,dsc_municipio_ibge,cod_ibge_cto,dsc_municipio_ibge_cto,posicao_gpon,concentrador,porta,caixa,posicao,lat_cto,long_cto,login,qtd_plano_internet,vlr_plano_internet,qtd_plano_sva,vlr_plano_sva,qtd_plano_hospedagem,vlr_plano_hospedagem,qtd_plano_radio,vlr_plano_radio,qtd_plano_telefone,vlr_plano_telefone,qtd_plano_mvno,vlr_plano_mvno,qtd_plano_outros,vlr_plano_outros,data_inclusao,data_aprovisionamento,data_atualizacao,vlr_cobrado_mes_atual,vlr_pago_mes_atual,tri_total_cobrado,tri_total_pago,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,faturado,flag_inadimplencia,status_pgto,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,razao_finalizados_vs_total_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,desconto_bloqueio_mes,tri_desconto_bloqueio_mes,flg_negociacao,tri_flg_negociacao,qtd_parcelas,parcela_atual,vlr_base,vlr_devido,qtd_bloqueio,tipo_churn,data_exclusao,tri_qtd_bloqueio
0,2025-04-01,base_final,1,0,0,0,0,6,2668640,385867,2014-01-01,2014-01-01,136,Fibra Home Combo,Fibra,T-G-25.600M.6000G,10,109.99,NaN,None,Cliente sem desconto,A,600000,300000,109.99,0.0,Turbo-G-600M,Fibra Home Combo 600M+ 2025,PF,Pedro Piazentin Neto,135.347.598-00,3552403,Sumaré,3552403,Sumaré,2420261,468,42008,13,105,-22.829052,-47.268673,6.1@desktop.com.br,1,109.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2025-01-24 08:10:50,2025-01-24 08:14:45,2025-10-24,109.99,109.99,438.03,438.03,1.0,0,146.01,0.0,0.0,-6.0,-4.0,1,0.0,antecipado,9.0,9.0,0.0,1.0,11.0,10.0,0.0,0.91,10.0,9.0,0.9,93.0,90.0,0.97,3.0,0.0,0.0,0.0,28.0,1.0,0.0,0.04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaT,NaN
1,2025-04-01,base_final,1,0,0,0,0,9,3147322,2091936,2006-03-03,2006-03-03,229,Fibra Home Combo,Fibra,T-G-24.400M.4000G,15,99.99,NaN,None,Cliente sem desconto,A,400000,200000,99.99,0.0,Turbo-G-400M,Fibra Home Combo 400M 2024 *,PF,Fabricio de Lima Moraes,121.668.688-27,3552403,Sumaré,3552403,Sumaré,2954956,558,52781,2483,18686,-22.828738,-47.211647,9.2@desktop.com.br,1,99.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2025-06-06 18:55:35,2025-06-06 18:52:58,2025-10-24,84.99,84.99,254.97,254.97,1.0,0,84.99,2.0,0.0,0.0,2.0,1,0.0,em_dia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,1.0,5.0,5.0,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaT,NaN
2,2025-04-01,base_final,1,0,0,0,0,12,2338539,1925310,2025-06-01,2005-03-18,241,Fibra Home Combo,Fibra,T-G-24.650M.4000G,20,114.99,NaN,None,Cliente sem desconto,A,650000,325000,114.99,0.0,Turbo-G-600M,Fibra Home Combo 650M 2024 *,PF,Rui Savitsky,619.721.268-49,3552403,Sumaré,3552403,Sumaré,281345,227,5175,340,2713,-22.799621,-47.216536,12.1@desktop.com.br,1,114.99,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,NaT,2024-09-25 16:09:32,2025-10-24,114.99,114.99,344.97,344.97,1.0,0,114.99,0.0,0.0,-10.0,-3.0,1,0.0,antecipado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

**Safras**: A base tem inforamação do **fechamento de abril (2025-04-01)** e do **fechamento de maio (2025-05-01)** - duas safras.

In [8]:
df['data_movimento'].drop_duplicates() #data_movimento é o último dia do mês de referência.

0        2025-04-01
976270   2025-05-01
Name: data_movimento, dtype: datetime64[ns]

In [5]:
#Volumetria
df.shape[0] #1.987.096

1987106

In [6]:
#Criar uma nova variável: razao_pgto_vs_cobrado_mes_atual
df['razao_pgto_vs_cobrado_mes_atual'] = np.where(
    df.vlr_cobrado_mes_atual == 0,  # Condição
    1,                               # Valor se verdadeiro
    df.vlr_pago_mes_atual / df.vlr_cobrado_mes_atual  # Valor se falso
)

# 2 - ETL

Tabela com as variáveis quantitativas.

In [7]:
#Filtrar as colunas relevantes
df_base = df[['data_movimento', 'num','data_cadastro', 'meses_base', 'download', 'upload', 'qtd_plano_internet', 'vlr_plano_internet', 
              'vlr_cobrado_mes_atual', 'vlr_pago_mes_atual', 'razao_pgto_vs_cobrado_mes_atual' , 'tri_total_cobrado', 
              'tri_total_pago', 'tri_razao_pgto_vs_cobrado','dias_inadimplencia_fatura_mais_antiga', 'tri_ticket_medio_pagamento', 
              'tri_freq_pgto_atraso', 'tri_valor_total_em_atraso', 'dias_atraso', 'tri_data_max_dias_atraso', 'qtd_total_mes_matrix',
              'qtd_finalizado_mes_matrix', 'qtd_finalizado_inatividade_mes_matrix', 'razao_finalizados_vs_total_mes_matrix',
              'qtd_total_trimestre_matrix', 'qtd_finalizado_trimestre_matrix', 'qtd_finalizado_inatividade_trimestre_matrix',
              'razao_finalizados_vs_total_trimestre_matrix', 'qtd_total_mes_caso', 'qtd_finalizado_mes_caso',
              'qtd_total_trimestre_caso', 'qtd_finalizado_trimestre_caso', 'razao_finalizados_vs_total_trimestre_caso',
              'qtd_total_mes_workorder', 'qtd_finalizado_mes_workorder', 'qtd_cancelado_mes_workorder', 'razao_finalizados_vs_total_mes_workorder',
              'qtd_total_trimestre_workorder', 'qtd_finalizado_trimestre_workorder', 'qtd_cancelado_trimestre_workorder',
              'razao_finalizados_vs_total_trimestre_workorder', 'tri_qtd_bloqueio', 'qtd_bloqueio','tipo_churn','data_exclusao']]    
#Visualização
df_base.head(3)

,data_movimento,num,data_cadastro,meses_base,download,upload,qtd_plano_internet,vlr_plano_internet,vlr_cobrado_mes_atual,vlr_pago_mes_atual,razao_pgto_vs_cobrado_mes_atual,tri_total_cobrado,tri_total_pago,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,tri_qtd_bloqueio,qtd_bloqueio,tipo_churn,data_exclusao
0,2025-04-01,6,2014-01-01,136,600000,300000,1,109.99,109.99,109.99,1.0,438.03,438.03,1.0,0,146.01,0.0,0.0,-6.0,-4.0,9.0,9.0,0.0,1.0,11.0,10.0,0.0,0.91,10.0,9.0,93.0,90.0,0.97,3.0,0.0,0.0,0.0,28.0,1.0,0.0,0.04,NaN,NaN,None,NaT
1,2025-04-01,9,2006-03-03,229,400000,200000,1,99.99,84.99,84.99,1.0,254.97,254.97,1.0,0,84.99,2.0,0.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,5.0,5.0,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaT
2,2025-04-01,12,2005-03-18,241,650000,325000,1,114.99,114.99,114.99,1.0,344.97,344.97,1.0,0,114.99,0.0,0.0,-10.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaT


## Churn

### Mensuração do alvo

A variável **flag_churn** indica se o cliente se tornou churn nos 4 meses após a safra de fechamento (janela de avaliação).

In [9]:
# Definindo as condições (os "WHEN" do CASE)
conditions = [
    # Condição 1: data_movimento = 2025-04-01 E data_exclusao até agosto
    (df_base['data_movimento'] == '2025-04-01') & (df_base['data_exclusao'].notna()) & (df_base['data_exclusao'] <= '2025-08-31'),
    
    # Condição 2: data_movimento = 2025-05-01 E data_exclusao até setembro
    (df_base['data_movimento'] == '2025-05-01') & (df_base['data_exclusao'].notna()) & (df_base['data_exclusao'] <= '2025-09-30')
]

# Definindo os valores para cada condição (os "THEN" do CASE)
# Se a primeira condição for verdadeira, o valor será 1.
# Se a segunda for verdadeira, o valor será 1.
choices = [1, 1]

# Definindo o valor padrão (o "ELSE" do CASE)
default_value = 0

# Criando a nova coluna 'flag_churn' usando np.select
df_base['flag_churn'] = np.select(conditions, choices, default=default_value)

### Tipo do churn

* Voluntário (V) ou Compulsório (C)

In [10]:
df_base['tipo_churn'].value_counts()

tipo_churn
C    131252
V    104384
Name: count, dtype: int64

In [11]:
#Criação da variável churn voluntário - flag_churn_vol
df_base['flag_churn_vol'] = ((df_base['flag_churn'] == 1) & (df_base['tipo_churn'] == 'V')).astype(int)
#Criação da variável churn compulsório - flag_churn_comp
df_base['flag_churn_comp'] = ((df_base['flag_churn'] == 1) & (df_base['tipo_churn'] == 'C')).astype(int)

In [12]:
#Visualização
df_base.head(3)

,data_movimento,num,data_cadastro,meses_base,download,upload,qtd_plano_internet,vlr_plano_internet,vlr_cobrado_mes_atual,vlr_pago_mes_atual,razao_pgto_vs_cobrado_mes_atual,tri_total_cobrado,tri_total_pago,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,tri_qtd_bloqueio,qtd_bloqueio,tipo_churn,data_exclusao,flag_churn,flag_churn_vol,flag_churn_comp
0,2025-04-01,6,2014-01-01,136,600000,300000,1,109.99,109.99,109.99,1.0,438.03,438.03,1.0,0,146.01,0.0,0.0,-6.0,-4.0,9.0,9.0,0.0,1.0,11.0,10.0,0.0,0.91,10.0,9.0,93.0,90.0,0.97,3.0,0.0,0.0,0.0,28.0,1.0,0.0,0.04,NaN,NaN,None,NaT,0,0,0
1,2025-04-01,9,2006-03-03,229,400000,200000,1,99.99,84.99,84.99,1.0,254.97,254.97,1.0,0,84.99,2.0,0.0,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,5.0,5.0,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaT,0,0,0
2,2025-04-01,12,2005-03-18,241,650000,325000,1,114.99,114.99,114.99,1.0,344.97,344.97,1.0,0,114.99,0.0,0.0,-10.0,-3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,NaT,0,0,0


In [13]:
##Salvar em parquet
df_base.to_parquet('base_arrumada_abril_maio_segmentacao.parquet', engine='fastparquet')

# 3 - Divisão dos dados por safra

**Abril**

In [14]:
abril = df_base.loc[df_base.data_movimento == '2025-04-01']
#Volumetria
abril.shape[0] #976.260

976270

In [15]:
#Unicidade no num_adm
abril['num'].drop_duplicates().shape[0] #976260

976260

In [16]:
#Taxa do churn
100 * abril['flag_churn'].value_counts(True) #7.884602%

flag_churn
0    92.115398
1     7.884602
Name: proportion, dtype: float64

In [17]:
#Taxa do churn voluntário
100 * abril['flag_churn_vol'].value_counts(True) #3.240087%

flag_churn_vol
0    96.759913
1     3.240087
Name: proportion, dtype: float64

In [18]:
#Taxa do churn compulsório
100 * abril['flag_churn_comp'].value_counts(True) #4.644514%

flag_churn_comp
0    95.355486
1     4.644514
Name: proportion, dtype: float64

**Maio**

In [19]:
maio = df_base.loc[df_base.data_movimento == '2025-05-01']
#Volumetria
maio.shape[0] #1.010.836

1010836

In [20]:
##Salvar em parquet
maio.to_parquet('base_teste_maio_segmentacao.parquet', engine='fastparquet')

# 4 - Construção do cluster

* **Nota**: Construir o modelo com base na safra de abril.

In [21]:
#Safra de interesse
abril = abril.fillna(0) #Trocar o missing por 0.

## 1. Avaliação do poder preditivo das variáveis

* **Aplicar o KS**: Cruzar as variáveis com o churn

In [22]:
# Cria uma lista com as variáveis de abril
colunas_var_quant = list(abril.columns.values)
# Elementos que você quer remover
elementos_para_remover = ['data_movimento', 'num', 'data_cadastro', 'tipo_churn', 'data_exclusao', 'flag_churn', 'flag_churn_comp', 'flag_churn_vol']
# List comprehension para filtrar a lista
colunas_var_quant = [coluna for coluna in colunas_var_quant if coluna not in elementos_para_remover] 

In [23]:
#Criação de um dicionário
dicionario = {}
#Aplicação de um laço para o cálculo do KS2 da lista de variáveis em colunas_var_quant
for coluna in colunas_var_quant:
   resultado = calcular_ks_2samp(abril,'flag_churn', coluna)
   dicionario[coluna] = resultado

In [24]:
# 1. Converter o dicionário em DataFrame
df_ks2 = pd.DataFrame(list(dicionario.items()), columns=['VARIAVEL', 'KS2'])

# 2. Ajustar KS2: multiplicar por 100 e arredondar para 2 casas decimais
df_ks2['KS2'] = round(df_ks2['KS2'] * 100, 2)

# 3. Converter os VALORES da coluna 'VARIAVEL' para MAIÚSCULAS
df_ks2['VARIAVEL'] = df_ks2['VARIAVEL'].str.upper()  # Esta linha faz a conversão

# 4. Visualizar o KS das variáveis
df_ks2.head(50)

,VARIAVEL,KS2
0,MESES_BASE,22.10
1,DOWNLOAD,5.39
2,UPLOAD,5.39
3,QTD_PLANO_INTERNET,0.34
4,VLR_PLANO_INTERNET,4.48
5,VLR_COBRADO_MES_ATUAL,9.69
6,VLR_PAGO_MES_ATUAL,52.08
7,RAZAO_PGTO_VS_COBRADO_MES_ATUAL,53.04
8,TRI_TOTAL_COBRADO,13.63
9,TRI_TOTAL_PAGO,44.82


## 2. Selecionar as variáveis

In [25]:
#Foram selecionadas as variáveis com maior preditivo para definir o total de clusters
df_model = abril[['meses_base', 'vlr_cobrado_mes_atual','vlr_pago_mes_atual','razao_pgto_vs_cobrado_mes_atual',
'tri_total_cobrado','tri_total_pago','tri_razao_pgto_vs_cobrado','dias_inadimplencia_fatura_mais_antiga',
'tri_ticket_medio_pagamento','tri_freq_pgto_atraso','tri_valor_total_em_atraso','dias_atraso',
'tri_data_max_dias_atraso','tri_qtd_bloqueio','qtd_bloqueio','razao_finalizados_vs_total_mes_matrix', 
'razao_finalizados_vs_total_trimestre_caso','razao_finalizados_vs_total_mes_workorder']]
df_model.info() #18 variáveis numéricas

<class 'pandas.core.frame.DataFrame'>
Index: 976270 entries, 0 to 976269
Data columns (total 18 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   meses_base                                 976270 non-null  int64  
 1   vlr_cobrado_mes_atual                      976270 non-null  float64
 2   vlr_pago_mes_atual                         976270 non-null  float64
 3   razao_pgto_vs_cobrado_mes_atual            976270 non-null  float64
 4   tri_total_cobrado                          976270 non-null  float64
 5   tri_total_pago                             976270 non-null  float64
 6   tri_razao_pgto_vs_cobrado                  976270 non-null  float64
 7   dias_inadimplencia_fatura_mais_antiga      976270 non-null  int64  
 8   tri_ticket_medio_pagamento                 976270 non-null  float64
 9   tri_freq_pgto_atraso                       976270 non-null  float64
 10  tri_valor_tot

In [26]:
df_model.head(3)

,meses_base,vlr_cobrado_mes_atual,vlr_pago_mes_atual,razao_pgto_vs_cobrado_mes_atual,tri_total_cobrado,tri_total_pago,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,tri_qtd_bloqueio,qtd_bloqueio,razao_finalizados_vs_total_mes_matrix,razao_finalizados_vs_total_trimestre_caso,razao_finalizados_vs_total_mes_workorder
0,136,109.99,109.99,1.0,438.03,438.03,1.0,0,146.01,0.0,0.0,-6.0,-4.0,0.0,0.0,1.0,0.97,0.0
1,229,84.99,84.99,1.0,254.97,254.97,1.0,0,84.99,2.0,0.0,0.0,2.0,0.0,0.0,0.0,1.00,0.0
2,241,114.99,114.99,1.0,344.97,344.97,1.0,0,114.99,0.0,0.0,-10.0,-3.0,0.0,0.0,0.0,0.00,0.0


##  3. Criação dos Clusters

**Normalização dos dados**

In [27]:
#Biblioteca da normalização
from sklearn.preprocessing import StandardScaler
#Instanciação de normalização
scaler = StandardScaler()
#Computa a média e o desvio-padrão para cada coluna
scaler.fit(df_model)
#Aplica a normalização (transformação Z-Score) e retorna o resultado um data frame
scaled_ds = pd.DataFrame(scaler.transform(df_model),columns= df_model.columns)

**Aplica o PCA**
* Reduzir a dimensão do conjunto de variáveis.
* Encontrar o menor número de componentes que retêm 85% da informação (variância) total das 17 variáveis.

**Encontrar o número ideal de componentes**

In [28]:
#Biblioteca do PCA
from sklearn.decomposition import PCA
# Treinar o PCA com o número total de features (se df_model tem 17 colunas)
pca = PCA().fit(scaled_ds)
# O array 'explained_variance_ratio_' mostra a proporção de variância de cada componente
# Você pode somar os valores para encontrar quantos componentes precisa para atingir, por exemplo, 85%
variancia_acumulada = np.cumsum(pca.explained_variance_ratio_)

# Encontre o índice onde a variância acumulada ultrapassa 0.85
k_ideal = np.where(variancia_acumulada >= 0.85)[0][0] + 1 

print(f"Número de componentes necessários para 85% de variância: {k_ideal}")

Número de componentes necessários para 85% de variância: 6


* **Entendimento**: Das 18 variáveis, 6 Componentes Principais (fatores) retém mais de 85% de toda a informação (variância) contida nos dados.

In [29]:
# Supondo que 'pca' é o seu objeto PCA TREINADO (PCA().fit(scaled_ds))

# 1. Obter a lista das colunas originais (Supondo que você a salvou corretamente)
# Utilize a lista que você usou para criar o scaled_ds
colunas_originais = df_model.columns.tolist() 

# 2. Extrair os pesos (loadings)
loadings = pca.components_

# 3. Criar um DataFrame de Loadings
loadings_df = pd.DataFrame(
    data=loadings,
    columns=colunas_originais,
    index=[f'PC_ {i+1}' for i in range(pca.n_components_)] 
)

# 4. Transpor o DataFrame para que as Variáveis sejam as Linhas
# (Mais fácil de ler: Var 1, Var 2, etc., têm pesos em PC1, PC2, etc.)
loadings_df_transposed = loadings_df.T 

print("Tabela de Loadings (Pesos de Cada Variável em Cada Componente):")
# USANDO PRINT SIMPLES (sem .to_markdown) para evitar o erro 'tabulate'
print(loadings_df_transposed)

# 5. Exibir a Variância Explicada (Confirmação da importância)
print("\nVariância explicada por cada componente (Importância):")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"PC_ {i+1}: {var*100:.2f}%")

Tabela de Loadings (Pesos de Cada Variável em Cada Componente):
                                              PC_ 1     PC_ 2     PC_ 3  \
meses_base                                 0.126252  0.013608 -0.083491   
vlr_cobrado_mes_atual                      0.302786  0.265735 -0.117532   
vlr_pago_mes_atual                         0.368673  0.049811 -0.034403   
razao_pgto_vs_cobrado_mes_atual            0.339599 -0.047874  0.011019   
tri_total_cobrado                          0.307697  0.252631 -0.111264   
tri_total_pago                             0.359547  0.114482 -0.043749   
tri_razao_pgto_vs_cobrado                  0.338116  0.012866  0.006795   
dias_inadimplencia_fatura_mais_antiga     -0.199405  0.381124 -0.187263   
tri_ticket_medio_pagamento                 0.347477  0.163738 -0.057860   
tri_freq_pgto_atraso                       0.135669  0.166894  0.297677   
tri_valor_total_em_atraso                 -0.192400  0.390283 -0.180216   
dias_atraso                         

**Identificando os PC's encontrados**
* **PC 1**: Fator de Valor e Fluxo de Caixa (39.74%) - Representa clientes com alta capacidade de pagamento e volume financeiro (o mais importante).
* **PC 2**: Fator de Risco e Gravidade da Inadimplência (21.41%) - Representa o histórico de atraso, o valor total em atraso e o tempo que a fatura fica inadimplente.
* **PC 3**: Fator de Problema e Bloqueio Operacional (12.80%) - Dominado por bloqueios e fechamento de tickets (WhatsApp/Protocolo Aberto). Indica clientes que dão problemas e são frequentemente bloqueados.
* **PC 4**: Fator de Demanda de Serviço e Solução (6.39%) - Dominado por razões de finalização de tickets (Ordem de Serviço/Protocolo Aberto). Representa o quão ativa a demanda do cliente por serviço é.
* **PC 5**: Fator de Longevidade (5.35%) - Fator puro e quase totalmente definido pelo tempo do cliente na empresa.
* **PC 6**: Fator de Atendimento/Ordem de Serviço (4.34%) - Indica forte separação entre clientes que têm alta taxa de tickets finalizados pelo WhatsApp versus taxa de ordem de serviço finalizada.

$$\mathbf{\text{Cluster}} = f(\mathbf{\text{Valor}}, \mathbf{\text{Risco}}, \mathbf{\text{Bloqueio}}, \mathbf{\text{Serviço}}, \mathbf{\text{Longevidade}}, \mathbf{\text{Atendimento}})$$

**PCAs criados**

**Aplicar o PCA**

In [31]:
# 1. Refazer o PCA com o número ideal de componentes
pca_final = PCA(n_components=6)
pca_final.fit(scaled_ds)

# 2. Criar o DataFrame final para o clustering
# Você terá 10 colunas que representam 85% da informação original
colunas_pca = [f"PC_{i+1}" for i in range(6)]
PCA_ds = pd.DataFrame(pca_final.transform(scaled_ds), columns=colunas_pca)

# PCA_ds está pronto para o K-Means!

In [32]:
PCA_ds.describe().T

,count,mean,std,min,25%,50%,75%,max
PC_1,976270.0,-5.962250e-17,2.601187,-16.108537,-0.023988,0.741073,1.284013,77.191299
PC_2,976270.0,0.000000e+00,1.908156,-6.897682,-0.797545,-0.306374,0.436842,39.120837
PC_3,976270.0,2.235844e-17,1.546504,-17.400368,-1.094736,-0.468840,0.893637,10.880779
PC_4,976270.0,4.471688e-17,1.234008,-5.759799,-0.684097,-0.185703,0.475576,27.127061
PC_5,976270.0,-2.981125e-17,0.953839,-7.636077,-0.688926,-0.196410,0.398512,8.811471
PC_6,976270.0,-1.490563e-17,0.859954,-24.085330,-0.357832,-0.102132,0.428922,6.174694


**Aplicar o K-Means**

1. Encontrar o número ótimo de clusters através do método da silhueta.
2. Utilizou-se uma amostra de 40 mil observações para definir o total de clusters.

In [33]:
from sklearn.metrics import silhouette_score

# 1. Definir o tamanho da amostra (ex: 40.000 observações)
sample_size = 40000 

# 2. Criar a amostra aleatória do PCA_ds
# O random_state garante que a amostra será a mesma a cada execução
PCA_ds_sample = PCA_ds.sample(n=sample_size, random_state=42)

melhor_k = 0
maior_silhueta = -1

print(f"Testando o Score de Silhueta em uma amostra de {sample_size} observações (k=4, 5, 6):")

# 3. Testar a faixa de interesse de negócio (k = 4, 5 e 6) na AMOSTRA
for k in range(4, 7):
    # n_init='auto' é o padrão recomendado no scikit-learn
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    
    # TREINA NA AMOSTRA
    kmeans.fit(PCA_ds_sample) 
    
    # Previsão e Cálculo da Silhueta também na AMOSTRA
    labels = kmeans.predict(PCA_ds_sample)
    score = silhouette_score(PCA_ds_sample, labels)
    
    print(f"K={k} -> Score de Silhueta (Amostra): {score:.4f}")
    
    if score > maior_silhueta:
        maior_silhueta = score
        melhor_k = k

print(f"\nO k ideal encontrado na amostra é: {melhor_k}") #O k ideal encontrado na amostra é: 4

Testando o Score de Silhueta em uma amostra de 40000 observações (k=4, 5, 6):
K=4 -> Score de Silhueta (Amostra): 0.3974
K=5 -> Score de Silhueta (Amostra): 0.3978
K=6 -> Score de Silhueta (Amostra): 0.4001

O k ideal encontrado na amostra é: 6


**Treinar o K-Means com k=4** 
* **Estratégia_1**: k = 4, 5 ou 6 não tem diferença. O ganho é marginal com k = 6 ou k = 5.
* **Estratégia**: Agrupar os cluster 2 e 3 num cluster só.

In [34]:
# 1. TREINAMENTO FINAL: K-Means com k=4
kmeans = KMeans(n_clusters=4, random_state=1935, n_init='auto') # n_init='auto' é o novo padrão
kmeans.fit(PCA_ds)
labels = kmeans.labels_
abril['cluster'] = labels

#Fundir os grupos 2 e 3 num só grupo
abril.loc[abril['cluster'] == 3, 'cluster'] = 2

# 2. AGREGAÇÃO DE MÚLTIPLAS MÉTRICAS (Volumetria, Total Churn, Taxa)
analise_risco = abril.groupby('cluster')['flag_churn'].agg(
    Volumetria = 'count',  
    Total_Churn = 'sum',   
    Taxa_Churn_Decimal = 'mean'    
).reset_index()

# 3. FORMATAR E CLASSIFICAR
analise_risco['Taxa_Churn (%)'] = (analise_risco['Taxa_Churn_Decimal'] * 100).round(2)

analise_risco = analise_risco.sort_values(by='Taxa_Churn_Decimal', ascending=False)

# 4. SELECIONAR E IMPRIMIR (Imprimindo o DataFrame diretamente)
analise_risco = analise_risco[['cluster', 'Volumetria', 'Total_Churn', 'Taxa_Churn (%)']]

print("\nAnálise de Volumetria e Risco de Churn por Cluster:")
print(analise_risco)


Análise de Volumetria e Risco de Churn por Cluster:
   cluster  Volumetria  Total_Churn  Taxa_Churn (%)
1        1       37578        35899           95.53
2        2      374526        20793            5.55
0        0      564166        20283            3.60


In [35]:
#Churn compulsório
resultado_compulsorio = (abril.groupby('cluster')['flag_churn_comp']
             .mean()
             .mul(100)
             .reset_index()
             .rename(columns={'flag_churn_comp': 'Churn Compulsório'}))
#Resultado
resultado_compulsorio

,cluster,Churn Compulsório
0,0,0.087208
1,1,94.752249
2,2,2.468453


In [36]:
#Churn voluntário
resultado_voluntario = (abril.groupby('cluster')['flag_churn_vol']
             .mean()
             .mul(100)
             .reset_index()
             .rename(columns={'flag_churn_comp': 'Churn Voluntário'}))
#Resultado
resultado_voluntario

,cluster,flag_churn_vol
0,0,3.508010
1,1,0.779712
2,2,3.083364


In [37]:
#Visualização
abril.head(3)

,data_movimento,num,data_cadastro,meses_base,download,upload,qtd_plano_internet,vlr_plano_internet,vlr_cobrado_mes_atual,vlr_pago_mes_atual,razao_pgto_vs_cobrado_mes_atual,tri_total_cobrado,tri_total_pago,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,tri_qtd_bloqueio,qtd_bloqueio,tipo_churn,data_exclusao,flag_churn,flag_churn_vol,flag_churn_comp,cluster
0,2025-04-01,6,2014-01-01,136,600000,300000,1,109.99,109.99,109.99,1.0,438.03,438.03,1.0,0,146.01,0.0,0.0,-6.0,-4.0,9.0,9.0,0.0,1.0,11.0,10.0,0.0,0.91,10.0,9.0,93.0,90.0,0.97,3.0,0.0,0.0,0.0,28.0,1.0,0.0,0.04,0.0,0.0,0,0,0,0,0,2
1,2025-04-01,9,2006-03-03,229,400000,200000,1,99.99,84.99,84.99,1.0,254.97,254.97,1.0,0,84.99,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,2.0,2.0,5.0,5.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0,0,0,0,0,0
2,2025-04-01,12,2005-03-18,241,650000,325000,1,114.99,114.99,114.99,1.0,344.97,344.97,1.0,0,114.99,0.0,0.0,-10.0,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0,0,0,0,0,0


In [38]:
lista = ['meses_base','vlr_pago_mes_atual', 'tri_total_pago', 'razao_pgto_vs_cobrado_mes_atual', 'tri_freq_pgto_atraso', 'tri_data_max_dias_atraso','qtd_total_trimestre_workorder','qtd_bloqueio']
abril.groupby('cluster')[lista].describe()

meses_base                                                      \
             count       mean        std  min   25%   50%   75%    max   
cluster                                                                  
0         564166.0  36.753073  34.576008  0.0  12.0  26.0  51.0  342.0   
1          37578.0  13.080712  16.945826  0.0   3.0   7.0  16.0  222.0   
2         374526.0  28.178628  29.223287  0.0   7.0  20.0  41.0  293.0   

        vlr_pago_mes_atual                                                     \
                     count        mean        std  min    25%     50%     75%   
cluster                                                                         
0                 564166.0  103.815333  23.977149  0.0  89.99  102.41  112.41   
1                  37578.0    0.481180   6.700107  0.0   0.00    0.00    0.00   
2                 374526.0   82.601515  49.252363  0.0  51.94   99.99  112.63   

                 tri_total_pago                                         \
             max          count        mean         std  min       25%   
cluster                                                                  
0         734.93       564166.0  302.224978   81.909903  0.0  269.9700   
1         148.07        37578.0   79.678079   83.309505  0.0    0.0000   
2        1867.05       374526.0  240.616557  148.305281  0.0  121.1825   

                                  razao_pgto_vs_cobrado_mes_atual            \
            50%      75%      max                           count      mean   
cluster                                                                       
0        305.07  334.800  2162.93                        564166.0  1.004265   
1         76.86  125.555   738.52                         37578.0  0.005267   
2        292.14  336.900  5624.88                        374526.0  0.805751   

                                                         tri_freq_pgto_atraso  \
              std  min  25%       50%       75%      max                count   
cluster                                                                         
0        0.050911  0.0  1.0  1.000000  1.020302  9.91200             564166.0   
1        0.069716  0.0  0.0  0.000000  0.000000  1.09941              37578.0   
2        0.411161  0.0  1.0  1.021002  1.024645  4.00000             374526.0   

                                                     tri_data_max_dias_atraso  \
             mean       std  min  25%  50%  75%  max                    count   
cluster                                                                         
0        1.018688  1.119407  0.0  0.0  1.0  2.0  3.0                 564166.0   
1        0.610384  0.720235  0.0  0.0  0.0  1.0  3.0                  37578.0   
2        1.651669  1.292280  0.0  0.0  2.0  3.0  3.0                 374526.0   

                                                                  \
               mean        std   min    25%    50%    75%    max   
cluster                                                            
0          3.623251  12.667812 -37.0   -2.0    1.0    8.0  461.0   
1        225.110863  22.919987  47.0  202.0  233.0  246.0  506.0   
2         13.148401  19.232087 -33.0    0.0   12.0   20.0  476.0   

        qtd_total_trimestre_workorder                                          \
                                count      mean       std  min  25%  50%  75%   
cluster                                                                         
0                            564166.0  0.019025  0.218915  0.0  0.0  0.0  0.0   
1                             37578.0  0.069402  0.434342  0.0  0.0  0.0  0.0   
2                            374526.0  0.325176  0.782856  0.0  0.0  0.0  0.0   

              qtd_bloqueio                                               
          max        count      mean       std  min  25%  50%  75%  max  
cluster                                                                  
0        23.0     564166.0  0.036463  0.187457  0.0  0.0  0.0  0

## Analytics

In [39]:
cluster_0 = abril.loc[abril.cluster == 0]
#Volumetria
cluster_0.shape[0] #564166

564166

In [40]:
cluster_0['tri_data_max_dias_atraso'].describe(percentiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

count    564166.000000
mean          3.623251
std          12.667812
min         -37.000000
10%          -5.000000
20%          -3.000000
30%          -1.000000
40%           0.000000
50%           1.000000
60%           3.000000
70%           6.000000
80%          10.000000
90%          13.000000
max         461.000000
Name: tri_data_max_dias_atraso, dtype: float64

In [41]:
cluster_0['tri_total_pago'].describe(percentiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

count    564166.000000
mean        302.224978
std          81.909903
min           0.000000
10%         222.740000
20%         259.650000
30%         271.830000
40%         294.970000
50%         305.070000
60%         319.430000
70%         329.970000
80%         343.190000
90%         381.100000
max        2162.930000
Name: tri_total_pago, dtype: float64

In [43]:
cluster_0['meses_base'].describe(percentiles=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])

count    564166.000000
mean         36.753073
std          34.576008
min           0.000000
10%           5.000000
20%           9.000000
30%          14.000000
40%          19.000000
50%          26.000000
60%          35.000000
70%          45.000000
80%          58.000000
90%          81.000000
max         342.000000
Name: meses_base, dtype: float64

**Público Premium**
* Público para realizar uma up sell ou cross sell.
* Derivado do Cluster 0.

In [48]:
cluster_premium = cluster_0.loc[ (cluster_0.meses_base >= 26) & (cluster_0.tri_total_pago >= 319) & (cluster_0.tri_data_max_dias_atraso <=0)]
#Volumetria
cluster_premium.shape[0] #70355

70355

In [49]:
#Taxa de churn deste grupo
cluster_premium['flag_churn'].value_counts(True) #0.029934

flag_churn
0    0.970066
1    0.029934
Name: proportion, dtype: float64

In [50]:
#Taxa de churn deste grupo
cluster_premium['flag_churn_comp'].value_counts(True) #0.000099

flag_churn_comp
0    0.999901
1    0.000099
Name: proportion, dtype: float64

In [51]:
#Taxa de churn deste grupo
cluster_premium['flag_churn_vol'].value_counts(True) #0.970166

flag_churn_vol
0    0.970166
1    0.029834
Name: proportion, dtype: float64

# Tabela final

In [52]:
import numpy as np

conditions = [
    abril['cluster'] == 1,
    abril['cluster'] == 2,
    (abril['cluster'] == 0) & (abril['meses_base'] >= 26) & 
    (abril['tri_total_pago'] >= 319) & (abril['tri_data_max_dias_atraso'] <= 0)
]

choices = [
    'alto_risco',
    'risco_medio',
    'cliente_premium'
]

abril['segmentacao'] = np.select(conditions, choices, default='risco_baixo')

In [53]:
#Visualização
abril.head(3)

,data_movimento,num,data_cadastro,meses_base,download,upload,qtd_plano_internet,vlr_plano_internet,vlr_cobrado_mes_atual,vlr_pago_mes_atual,razao_pgto_vs_cobrado_mes_atual,tri_total_cobrado,tri_total_pago,tri_razao_pgto_vs_cobrado,dias_inadimplencia_fatura_mais_antiga,tri_ticket_medio_pagamento,tri_freq_pgto_atraso,tri_valor_total_em_atraso,dias_atraso,tri_data_max_dias_atraso,qtd_total_mes_matrix,qtd_finalizado_mes_matrix,qtd_finalizado_inatividade_mes_matrix,razao_finalizados_vs_total_mes_matrix,qtd_total_trimestre_matrix,qtd_finalizado_trimestre_matrix,qtd_finalizado_inatividade_trimestre_matrix,razao_finalizados_vs_total_trimestre_matrix,qtd_total_mes_caso,qtd_finalizado_mes_caso,qtd_total_trimestre_caso,qtd_finalizado_trimestre_caso,razao_finalizados_vs_total_trimestre_caso,qtd_total_mes_workorder,qtd_finalizado_mes_workorder,qtd_cancelado_mes_workorder,razao_finalizados_vs_total_mes_workorder,qtd_total_trimestre_workorder,qtd_finalizado_trimestre_workorder,qtd_cancelado_trimestre_workorder,razao_finalizados_vs_total_trimestre_workorder,tri_qtd_bloqueio,qtd_bloqueio,tipo_churn,data_exclusao,flag_churn,flag_churn_vol,flag_churn_comp,cluster,segmentacao
0,2025-04-01,6,2014-01-01,136,600000,300000,1,109.99,109.99,109.99,1.0,438.03,438.03,1.0,0,146.01,0.0,0.0,-6.0,-4.0,9.0,9.0,0.0,1.0,11.0,10.0,0.0,0.91,10.0,9.0,93.0,90.0,0.97,3.0,0.0,0.0,0.0,28.0,1.0,0.0,0.04,0.0,0.0,0,0,0,0,0,2,risco_medio
1,2025-04-01,9,2006-03-03,229,400000,200000,1,99.99,84.99,84.99,1.0,254.97,254.97,1.0,0,84.99,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,2.0,2.0,5.0,5.0,1.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0,0,0,0,0,0,risco_baixo
2,2025-04-01,12,2005-03-18,241,650000,325000,1,114.99,114.99,114.99,1.0,344.97,344.97,1.0,0,114.99,0.0,0.0,-10.0,-3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0,0,0,0,0,0,cliente_premium


In [54]:
resultados_abril_final = abril[['data_cadastro', 'num', 'segmentacao', 'flag_churn', 'flag_churn_vol', 'flag_churn_comp']]
#Visualização
resultados_abril_final.head(3)

,data_cadastro,num,segmentacao,flag_churn,flag_churn_vol,flag_churn_comp
0,2014-01-01,6,risco_medio,0,0,0
1,2006-03-03,9,risco_baixo,0,0,0
2,2005-03-18,12,cliente_premium,0,0,0


In [55]:
##Salvar em parquet
resultados_abril_final.to_parquet('abril_escoragem.parquet', engine='fastparquet')

In [56]:
import joblib

# Supondo que você tem esses objetos já treinados:
# scaler = StandardScaler().fit(df_model)
# pca = PCA(n_components=4).fit(scaled_ds)
# kmeans_final = KMeans(n_clusters=4, ...).fit(PCA_ds)

# SALVANDO CADA COMPONENTE DO PIPELINE

# 1. Salvar o Padronizador
joblib.dump(scaler, 'scaler_segmentacao.pkl')

# 2. Salvar o PCA
joblib.dump(pca, 'pca_segmentacao.pkl')

# 3. Salvar o K-Means (O modelo que gera 4 clusters)
joblib.dump(kmeans, 'kmeans_4_clusters.pkl')

print("Todos os componentes do pipeline foram salvos com sucesso em arquivos .pkl")

Todos os componentes do pipeline foram salvos com sucesso em arquivos .pkl
